# Part 2：加入 Workload Metadata

## 研究問題

batch、image size、AMP 與 interaction terms 能否補足固定跨 GPU scaling？

## 目前結論

最佳 combined Huber interactions：MAPE 3.75%。仍使用來源真實 runtime，且假設 metadata 可得，因此不符合最嚴格黑箱部署。

> 本 Notebook 新增於既有研究產物之外，不會取代原始 Python scripts 或 CSV／Markdown 報告。

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display


def find_project_root() -> Path:
    """Find the repository whether Jupyter starts at the repo or notebook folder."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "analysis").is_dir() and (candidate / "analysis_outputs").is_dir():
            return candidate
    raise FileNotFoundError("找不到專案根目錄；請從本 repository 內啟動 Jupyter。")


PROJECT_ROOT = find_project_root()
OUTPUT_ROOT = PROJECT_ROOT / "analysis_outputs"
RUN_EXPENSIVE = False  # 明確改成 True 才會重跑分析腳本。


def project_path(relative: str) -> Path:
    return PROJECT_ROOT / relative


def load_csv(relative: str, **kwargs) -> pd.DataFrame:
    path = project_path(relative)
    if not path.exists():
        raise FileNotFoundError(f"缺少資料：{path}")
    return pd.read_csv(path, **kwargs)


def show_markdown(relative: str) -> None:
    path = project_path(relative)
    display(Markdown(path.read_text(encoding="utf-8")))


def inspect_inputs(paths) -> pd.DataFrame:
    rows = []
    for relative in paths:
        path = project_path(relative)
        rows.append({
            "path": relative,
            "exists": path.exists(),
            "type": "directory" if path.is_dir() else "file",
            "size_mib": round(path.stat().st_size / 1024**2, 3) if path.is_file() else None,
        })
    return pd.DataFrame(rows)


def run_commands(commands) -> None:
    """Show commands by default; run only after explicit opt-in."""
    for args in commands:
        command = [sys.executable, *args]
        print(" ".join(map(str, command)))
        if RUN_EXPENSIVE:
            subprocess.run(command, cwd=PROJECT_ROOT, check=True)


print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"RUN_EXPENSIVE = {RUN_EXPENSIVE}")

## 實驗步驟

1. 從實驗設定建立 workload metadata。
2. 與 source runtime、target label 合併。
3. 在 outer LOOW 內做模型與超參數選擇。
4. 比較 Ridge、Huber、main effects 與 interactions。
5. 檢查兩個方向的 OOF error。

## 輸入與 provenance

下列 cell 只檢查輸入是否存在，不修改資料。

In [ ]:
INPUTS = [
    "analysis_outputs/ground_truth.csv",
    "analysis_outputs/workload_features.csv",
    "analysis_outputs/cross_device_samples.csv",
]
display(inspect_inputs(INPUTS))

## 重現入口

下列命令依現有 pipeline 排列。預設只列印；將 RUN_EXPENSIVE 明確改成 True 才會執行。

In [ ]:
COMMANDS = [
    ["analysis/build_workload_features.py"],
    ["analysis/run_workload_models.py"],
]
run_commands(COMMANDS)

## 結果表

In [ ]:
METRICS_PATH = "analysis_outputs/model_experiments/part2_workload_models/metrics.csv"
PREDICTIONS_PATH = "analysis_outputs/model_experiments/part2_workload_models/predictions.csv"

metrics = load_csv(METRICS_PATH)
sort_column = "mape_percent" if "mape_percent" in metrics.columns else "mean_period_ape_percent"
display(metrics.sort_values(sort_column).reset_index(drop=True))

if "direction" in metrics.columns and "model" in metrics.columns:
    plot_data = metrics.sort_values(sort_column).groupby("direction", as_index=False).first()
    ax = plot_data.plot.bar(x="direction", y=sort_column, legend=False, figsize=(8, 4))
    ax.set_ylabel(sort_column)
    ax.set_title("各方向最佳列（依表內指標排序）")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

## OOF 預測與 worst cases

顯示現有 prediction table 中 APE 最大的列，並提供 actual-vs-predicted 圖。

In [ ]:
METRICS_PATH = "analysis_outputs/model_experiments/part2_workload_models/metrics.csv"
PREDICTIONS_PATH = "analysis_outputs/model_experiments/part2_workload_models/predictions.csv"

predictions = load_csv(PREDICTIONS_PATH)
display(predictions.sort_values("ape_percent", ascending=False).head(15).reset_index(drop=True))

required = {"actual_ms", "predicted_ms", "ape_percent"}
if required.issubset(predictions.columns):
    view = predictions.copy()
    if "deployment_valid" in view.columns:
        view = view[view["deployment_valid"].astype(str).str.lower().eq("true")]
    ax = view.plot.scatter(x="actual_ms", y="predicted_ms", c="ape_percent", cmap="viridis", figsize=(6, 5))
    low = min(view["actual_ms"].min(), view["predicted_ms"].min())
    high = max(view["actual_ms"].max(), view["predicted_ms"].max())
    ax.plot([low, high], [low, high], "--", color="black", linewidth=1)
    ax.set_title("OOF prediction：actual vs predicted")
    plt.tight_layout()
    plt.show()

## 既有最終報告快照

> 來源：`analysis_outputs/model_experiments/part2_workload_models/report.md`。這是建立 Notebook 時嵌入的結果快照；重新執行分析後，請以磁碟上的最新報告為準。

# Part 2: workload-aware regularized models

Outer evaluation is leave-one-workload-out. Scaling and hyperparameter selection use only the outer training fold; selection uses an inner leave-one-workload-out loop.

| Direction | Model | MAE ms | RMSE ms | MAPE | Median APE | Max APE | R2 |
|---|---|---:|---:|---:|---:|---:|---:|
| RTX4090_to_RTX5090 | huber_interactions_log_target | 1.117 | 1.613 | 3.11% | 1.48% | 11.29% | 0.9930 |
| RTX4090_to_RTX5090 | ridge_main_log_target | 1.310 | 1.813 | 3.42% | 2.50% | 13.01% | 0.9911 |
| RTX4090_to_RTX5090 | ridge_interactions_log_ratio | 1.373 | 1.826 | 3.56% | 2.36% | 12.94% | 0.9910 |
| RTX4090_to_RTX5090 | elasticnet_interactions_log_target | 1.380 | 1.829 | 3.62% | 2.40% | 12.85% | 0.9910 |
| RTX4090_to_RTX5090 | ridge_interactions_log_target | 1.434 | 1.870 | 3.75% | 2.03% | 13.05% | 0.9905 |
| RTX5090_to_RTX4090 | ridge_main_log_target | 2.046 | 2.833 | 4.27% | 2.43% | 12.92% | 0.9864 |
| RTX5090_to_RTX4090 | huber_interactions_log_target | 1.933 | 2.421 | 4.40% | 2.93% | 12.48% | 0.9900 |
| RTX5090_to_RTX4090 | elasticnet_interactions_log_target | 2.479 | 3.761 | 4.80% | 3.46% | 12.60% | 0.9760 |
| RTX5090_to_RTX4090 | ridge_interactions_log_target | 2.300 | 2.993 | 4.85% | 3.87% | 12.60% | 0.9848 |
| RTX5090_to_RTX4090 | ridge_interactions_log_ratio | 2.678 | 3.815 | 5.12% | 3.94% | 12.58% | 0.9753 |
| both_directions_combined | huber_interactions_log_target | 1.525 | 2.057 | 3.75% | 2.36% | 12.48% | 0.9913 |
| both_directions_combined | ridge_main_log_target | 1.678 | 2.379 | 3.85% | 2.48% | 13.01% | 0.9884 |
| both_directions_combined | elasticnet_interactions_log_target | 1.930 | 2.957 | 4.21% | 2.84% | 12.85% | 0.9821 |
| both_directions_combined | ridge_interactions_log_target | 1.867 | 2.496 | 4.30% | 3.08% | 13.05% | 0.9872 |
| both_directions_combined | ridge_interactions_log_ratio | 2.026 | 2.991 | 4.34% | 3.04% | 12.94% | 0.9817 |

## Change relative to the best Part 1 baseline

| Direction | Part 1 best | Part 1 MAPE | Part 2 best | Part 2 MAPE | Relative change |
|---|---|---:|---|---:|---:|
| RTX4090_to_RTX5090 | log_source_ols | 4.10% | huber_interactions_log_target | 3.11% | -24.2% |
| RTX5090_to_RTX4090 | global_ratio_median | 4.23% | ridge_main_log_target | 4.27% | +1.0% |

## Hyperparameter selection frequency

- RTX4090_to_RTX5090 / elasticnet_interactions_log_target: `{"model__alpha": 0.001, "model__l1_ratio": 0.9}`: 21; `{"model__alpha": 0.001, "model__l1_ratio": 0.1}`: 1; `{"model__alpha": 0.01, "model__l1_ratio": 0.1}`: 1; `{"model__alpha": 0.001, "model__l1_ratio": 0.5}`: 1
- RTX4090_to_RTX5090 / huber_interactions_log_target: `{"model__alpha": 1.0, "model__epsilon": 1.1}`: 13; `{"model__alpha": 0.1, "model__epsilon": 1.1}`: 8; `{"model__alpha": 1.0, "model__epsilon": 1.35}`: 1; `{"model__alpha": 0.01, "model__epsilon": 1.1}`: 1; `{"model__alpha": 0.0, "model__epsilon": 1.1}`: 1
- RTX4090_to_RTX5090 / ridge_interactions_log_ratio: `{"model__alpha": 1.0}`: 22; `{"model__alpha": 0.1}`: 2
- RTX4090_to_RTX5090 / ridge_interactions_log_target: `{"model__alpha": 0.1}`: 24
- RTX4090_to_RTX5090 / ridge_main_log_target: `{"model__alpha": 0.01}`: 12; `{"model__alpha": 0.0}`: 8; `{"model__alpha": 0.1}`: 4
- RTX5090_to_RTX4090 / elasticnet_interactions_log_target: `{"model__alpha": 0.01, "model__l1_ratio": 0.5}`: 15; `{"model__alpha": 0.001, "model__l1_ratio": 0.9}`: 6; `{"model__alpha": 0.01, "model__l1_ratio": 0.9}`: 2; `{"model__alpha": 0.001, "model__l1_ratio": 0.1}`: 1
- RTX5090_to_RTX4090 / huber_interactions_log_target: `{"model__alpha": 1.0, "model__epsilon": 1.35}`: 15; `{"model__alpha": 1.0, "model__epsilon": 1.1}`: 4; `{"model__alpha": 0.0, "model__epsilon": 1.1}`: 3; `{"model__alpha": 0.01, "model__epsilon": 1.35}`: 1; `{"model__alpha": 0.1, "model__epsilon": 1.1}`: 1
- RTX5090_to_RTX4090 / ridge_interactions_log_ratio: `{"model__alpha": 1.0}`: 12; `{"model__alpha": 100.0}`: 10; `{"model__alpha": 0.1}`: 1; `{"model__alpha": 0.0}`: 1
- RTX5090_to_RTX4090 / ridge_interactions_log_target: `{"model__alpha": 0.1}`: 16; `{"model__alpha": 0.01}`: 7; `{"model__alpha": 0.0}`: 1
- RTX5090_to_RTX4090 / ridge_main_log_target: `{"model__alpha": 0.0}`: 22; `{"model__alpha": 0.01}`: 1; `{"model__alpha": 0.1}`: 1

## Worst workloads for the best Part 2 model

### RTX4090_to_RTX5090: `huber_interactions_log_target`

| Workload | Source ms | Actual ms | Predicted ms | APE |
|---|---:|---:|---:|---:|
| C10 | 49.200 | 39.302 | 43.738 | 11.29% |
| C01 | 30.278 | 29.060 | 25.995 | 10.55% |
| C17 | 38.002 | 31.537 | 34.437 | 9.20% |
| C15 | 38.872 | 33.221 | 36.206 | 8.99% |
| C09 | 31.867 | 30.749 | 28.686 | 6.71% |

### RTX5090_to_RTX4090: `ridge_main_log_target`

| Workload | Source ms | Actual ms | Predicted ms | APE |
|---|---:|---:|---:|---:|
| C01 | 29.060 | 30.278 | 34.189 | 12.92% |
| C10 | 39.302 | 49.200 | 43.020 | 12.56% |
| C17 | 31.537 | 38.002 | 33.856 | 10.91% |
| C15 | 33.221 | 38.872 | 35.455 | 8.79% |
| C09 | 30.749 | 31.867 | 34.552 | 8.42% |


## 解讀檢查清單

- 區分 OOF evaluation 與重新用全部資料訓練的 production bundle。
- 不把 target runtime、oracle marker 或 future information 當作 deployment feature。
- 報告方向別結果、combined summary 與 worst cases。
- 結論只適用於目前資料、GPU 節點與 workload family。